In [1]:
!pip install numpy pandas scikit-learn nltk joblib

In [2]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import nltk

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('punkt')

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Setup completed successfully!")

Setup completed successfully!


[nltk_data] Downloading package stopwords to /home/aryan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/aryan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
DATASET_PATH = '../data/tweets.csv'

column_names = ['target', 'id', 'date', 'flag', 'user', 'text']

print("Loading dataset...")
df = pd.read_csv(DATASET_PATH, encoding='ISO-8859-1', names=column_names)

df = df[['target', 'text']]

df['target'] = df['target'].replace(4, 1)

print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:")
print(df['target'].value_counts())
df.head()

Loading dataset...
Dataset Shape: (1600000, 2)

Class Distribution:
target
0    800000
1    800000
Name: count, dtype: int64


,target,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [4]:
port_stem = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(content: str) -> str:
    if not isinstance(content, str):
        return ""
    cleaned = re.sub(r"http\S+|@\S+|#\S+|[^\w\s]", " ", content)
    cleaned = cleaned.lower()
    
    # Stemming & stopword removal
    words = cleaned.split()
    stemmed = [port_stem.stem(word) for word in words if word not in stop_words]
    
    return ' '.join(stemmed)

print("Preprocessing dataset... (this may take 1-2 minutes for large datasets)")
df['cleaned_text'] = df['text'].apply(preprocess_text)
print("Preprocessing finished!")
df[['text', 'cleaned_text', 'target']].head()

Preprocessing dataset... (this may take 1-2 minutes for large datasets)
Preprocessing finished!


,text,cleaned_text,target
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww bummer shoulda got david carr third day,0
1,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...,0
2,@Kenichan I dived many times for the ball. Man...,dive mani time ball manag save 50 rest go bound,0
3,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire,0
4,"@nationwideclass no, it's not behaving at all....",behav mad see,0


In [5]:
X = df['cleaned_text'].values
Y = df['target'].values

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

print("Fitting TF-IDF Vectorizer...")
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Train feature matrix shape: {X_train_vec.shape}")

Fitting TF-IDF Vectorizer...
Train feature matrix shape: (1280000, 10000)


In [6]:
print("Training Logistic Regression model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, Y_train)

# Accuracy scores
train_preds = model.predict(X_train_vec)
test_preds = model.predict(X_test_vec)

print(f"\nTraining Accuracy: {accuracy_score(Y_train, train_preds) * 100:.2f}%")
print(f"Testing Accuracy:  {accuracy_score(Y_test, test_preds) * 100:.2f}%")
print("\nClassification Report:\n", classification_report(Y_test, test_preds, target_names=['NEGATIVE', 'POSITIVE']))

Training Logistic Regression model...

Training Accuracy: 78.10%
Testing Accuracy:  77.74%

Classification Report:
               precision    recall  f1-score   support

    NEGATIVE       0.79      0.75      0.77    160000
    POSITIVE       0.76      0.80      0.78    160000

    accuracy                           0.78    320000
   macro avg       0.78      0.78      0.78    320000
weighted avg       0.78      0.78      0.78    320000



In [8]:
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. Preprocess & Split Data
df['cleaned_text'] = df['text'].apply(preprocess_text)

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# 2. Vectorization (N-gram range 1-3)
vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=2
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 3. LinearSVC with Probability Calibration
print("Training Calibrated LinearSVC model...")
base_svm = LinearSVC(
    C=1.0, 
    class_weight='balanced', 
    random_state=42, 
    max_iter=2000
)

# CalibratedClassifierCV enables model.predict_proba()
model = CalibratedClassifierCV(estimator=base_svm, cv=3)
model.fit(X_train_vec, y_train)

# 4. Evaluation
y_pred = model.predict(X_test_vec)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 5. Save Model Artifacts
with open('../models/sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Saved updated LinearSVC 'sentiment_model.pkl' and 'vectorizer.pkl' successfully!")

Training Calibrated LinearSVC model...
Accuracy: 0.7816

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.76      0.78    160000
           1       0.77      0.80      0.79    160000

    accuracy                           0.78    320000
   macro avg       0.78      0.78      0.78    320000
weighted avg       0.78      0.78      0.78    320000

Saved updated LinearSVC 'sentiment_model.pkl' and 'vectorizer.pkl' successfully!
